# Lab 7 · Streamlit Demo + Gradio Annotation

Terkait: **Bab 07 · Alat Pendukung Ringan untuk Riset**

## Tujuan
1. Bangun `demo/app.py` dengan Streamlit: upload gambar → prediksi + top-3.
2. Bangun `demo/annotate.py` dengan Gradio untuk re-annotate sampel mencurigakan.
3. Plot ablation hasil Lab 2 dengan error bar yang jujur.

## Prasyarat
```bash
pip install -e '.[demo]'
```

## Checklist
- [ ] `demo/app.py` dapat dijalankan dengan `streamlit run demo/app.py`.
- [ ] Model di-load *satu kali* (`@st.cache_resource`) — tidak reload per request.
- [ ] Top-3 prediksi tampil dengan confidence.
- [ ] `demo/annotate.py` menyimpan anotasi ke CSV per klik.
- [ ] Plot ablation Lab 2 dengan error bar disimpan ke `experiments/plots/`.
- [ ] Plot dapat dibaca oleh orang yang tidak tahu konteks (standalone).

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/muhammad-zainal-muttaqin/ModulePembelajaran.git'
    REPO_DIR = '/content/ModulePembelajaran'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        print('Repo sudah ada, skip clone.')

    %cd {REPO_DIR}/ModulePembelajaran/template_repo
    !pip install -e '.[demo]'
    print('\nSetup Colab selesai. Working dir:', os.getcwd())
else:
    print('Environment lokal. Lewati setup Colab.')

## 0. Setup

### Lokal
```bash
pip install -e '.[demo]'
```

### Google Colab
Jalankan sel kode **0a. Setup Colab** di bawah ini. Setelah selesai, lanjut ke **1. Plot ablation**.

## 1. Plot ablation dari hasil Lab 2 (wajib dikerjakan di notebook)

Ini bagian yang dikerjakan di notebook — sisanya (Streamlit, Gradio) dikerjakan di file terpisah.

In [ ]:
results_path = Path('experiments/results.csv')

In [ ]:
output_dir = Path('experiments/plots')

## 2. Kerangka `demo/app.py` (Streamlit)

Kerjakan di file terpisah `demo/app.py`. Jalankan dengan:
```bash
streamlit run demo/app.py
```

Salin template di bawah ke `demo/app.py` dan lengkapi bagian TODO.

In [ ]:
annotate_path = Path('demo') / 'annotate.py'

## 3. Kerangka `demo/annotate.py` (Gradio)

Untuk re-annotate 30 sampel mencurigakan dari test set.

In [ ]:
annotate_code = '''
"""demo/annotate.py - Gradio annotation UI untuk re-labeling sampel.

Jalankan: python demo/annotate.py
Output: demo/annotations.csv (append per klik)
"""
import csv
import sys
from pathlib import Path
from datetime import datetime

import gradio as gr
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image

sys.path.insert(0, str(Path(__file__).parent.parent))
from src.models import build_model
from src.utils import load_config
from src.data import build_loaders

CLASSES = [\'airplane\', \'automobile\', \'bird\', \'cat\', \'deer\',
           \'dog\', \'frog\', \'horse\', \'ship\', \'truck\']
CSV_PATH = Path(\'demo/annotations.csv\')

# Load model dan ambil sampel paling confident-salah
def get_suspect_samples(n=30):
    ckpt_path = Path(\'experiments/baseline_seed42/ckpt_best.pt\')
    if not ckpt_path.exists():
        return []
    ckpt = torch.load(ckpt_path, map_location=\'cpu\')
    model = build_model(ckpt[\'config\'][\'model\'])
    model.load_state_dict(ckpt[\'model_state\'])
    model.eval()
    
    cfg = ckpt[\'config\']
    _, _, test_loader = build_loaders(cfg[\'data\'])
    
    samples = []
    with torch.no_grad():
        for x, y in test_loader:
            y = torch.as_tensor(y).long().view(-1)
            probs = torch.softmax(model(x), dim=1)
            preds = probs.argmax(dim=1)
            wrong = preds != y
            confs = probs.max(dim=1).values
            for i in range(len(y)):
                if wrong[i]:
                    samples.append((x[i], y[i].item(), preds[i].item(), confs[i].item()))
    
    samples.sort(key=lambda s: s[3], reverse=True)
    return samples[:n]

samples = get_suspect_samples()
sample_idx = [0]  # mutable index

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

def to_pil(tensor):
    img = (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    return Image.fromarray((img * 255).astype(np.uint8)).resize((128, 128))

def load_current():
    if not samples:
        return None, "Tidak ada sampel", "?"
    i = sample_idx[0]
    x, true_label, pred_label, conf = samples[i]
    img = to_pil(x)
    info = f"Sample {i+1}/{len(samples)} | True: {CLASSES[true_label]} | Pred: {CLASSES[pred_label]} ({conf*100:.1f}%)"
    return img, info, CLASSES[true_label]

def save_annotation(new_label, notes):
    i = sample_idx[0]
    x, true_label, pred_label, conf = samples[i]
    CSV_PATH.parent.mkdir(exist_ok=True)
    write_header = not CSV_PATH.exists()
    with open(CSV_PATH, \'a\', newline=\'\', encoding=\'utf-8\') as f:
        writer = csv.DictWriter(f, fieldnames=[\'timestamp\', \'sample_idx\', \'original_label\', \'model_pred\', \'new_label\', \'notes\'])
        if write_header:
            writer.writeheader()
        writer.writerow({
            \'timestamp\': datetime.now().isoformat(),
            \'sample_idx\': i,
            \'original_label\': CLASSES[true_label],
            \'model_pred\': CLASSES[pred_label],
            \'new_label\': new_label,
            \'notes\': notes or \'\',
        })
    sample_idx[0] = min(sample_idx[0] + 1, len(samples) - 1)
    return load_current()

img, info, default_label = load_current()

with gr.Blocks(title=\'Annotation Tool\') as demo:
    gr.Markdown(\'## Re-Annotation: Confident-Wrong Predictions\')
    with gr.Row():
        image_out = gr.Image(value=img, label=\'Gambar\')
        with gr.Column():
            info_text = gr.Textbox(value=info, label=\'Info Sampel\', interactive=False)
            label_radio = gr.Radio(choices=CLASSES, value=default_label, label=\'Label yang benar?\')
            notes_input = gr.Textbox(label=\'Catatan (opsional)\')
            save_btn = gr.Button(\'Simpan dan lanjut →\', variant=\'primary\')
    
    save_btn.click(save_annotation, inputs=[label_radio, notes_input],
                  outputs=[image_out, info_text, label_radio])

if __name__ == \'__main__\':
    demo.launch()
'''

annotate_path = demo_dir / 'annotate.py'
if not annotate_path.exists():
    annotate_path.write_text(annotate_code)
    print(f'Dibuat: {annotate_path}')
    print('Jalankan dengan: python demo/annotate.py')
else:
    print(f'{annotate_path} sudah ada.')

## 4. Verifikasi demo

Setelah `demo/app.py` dijalankan, verifikasi hal-hal berikut:

**Checklist verifikasi manual:**

- [ ] `streamlit run demo/app.py` → halaman terbuka tanpa error
- [ ] Upload gambar kucing JPEG → prediksi muncul
- [ ] Refresh halaman → model tidak di-reload (loading cepat = cache berhasil)
- [ ] Upload gambar yang bukan CIFAR category (misal: mobil dari depan) → prediksi masuk akal?
- [ ] `python demo/annotate.py` → UI terbuka, klik Simpan → `demo/annotations.csv` berisi 1 baris

## 5. Refleksi

1. Saat menunjukkan demo ke orang lain, apa satu pertanyaan yang pasti mereka tanyakan yang belum kamu siapkan jawabannya di UI?

2. Plot ablation yang kamu buat: apakah seseorang yang tidak tahu konteks Lab 2 bisa membaca dan memahami pesannya? Tunjukkan ke satu teman dan catat satu hal yang membingungkan mereka.

3. Prinsip Bab 07 menyebut demo yang baik harus "membuktikan sesuatu". Apa *satu hal spesifik* yang demo ini buktikan atau bantah?

### Jawaban Refleksi

**1. Pertanyaan yang belum disiapkan:**
> *[tulis di sini]*

**2. Umpan balik dari teman tentang plot:**
> *[tulis di sini]*

**3. Apa yang demo ini buktikan:**
> *[tulis di sini]*